# F1 Strategy Manager - Data Engineering Pipeline

This notebook takes the lap master dataset produced in N01-N02 and the circuit cluster assignments from N03, and engineers the full feature set required by the ML models in Phase 3.

**Input:**
- `data/raw/*/laps.parquet` — 52,340 laps across 47 GPs (2023–2024), 37 columns
- `data/raw/*/intervals.parquet` — 1.1M interval samples for gap features
- `data/processed/circuit_clustering/circuit_clusters_k4.parquet` — N03 cluster mapping

**Output:**
- `data/processed/laps_featured.parquet` — enriched dataset (~53 columns), ready for ML
- `data/processed/feature_manifest.json` — column registry for downstream models

**Features added (35 new columns):**
| Step | Group | Examples |
|------|-------|---------|
| 2 | Fuel-corrected degradation | `FuelAdjustedLapTime`, `FuelAdjustedDegPercent` |
| 3 | Sequential lap features | `Prev_LapTime`, `LapTime_Delta`, `LapTime_Trend` |
| 4 | Stint degradation rate | `DegradationRate`, `CumulativeDeg` |
| 5 | Weather conditions | `AirTemp`, `TrackTemp`, `Humidity`, `Rainfall` |
| 6 | Race context | `race_phase`, `laps_remaining`, `track_status_clean` |
| 7 | Circuit-cluster | `Cluster`, `lap_time_vs_cluster_mean` |
| 7.5 | Temporal normalization | `lap_time_pct_of_race_fastest` |

> **Note:** All features are computable at inference time — no future data leakage.
> Train/val split: 2023 season (train) / 2024 season (val+test), defined in Step 8.

---

In [1]:
## Data Manipulation
import pandas as pd
import numpy as np

## Visualization
import matplotlib.pyplot as plt
import seaborn as sns

## Statistics (para DegradationRate con numpy.polyfit)
from scipy import stats

## File Handling
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.3f}'.format)

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10


In [3]:
# Find repository root (where .git folder is)
current_path = Path.cwd()
while not (current_path / ".git").exists() and current_path != current_path.parent:
    current_path = current_path.parent
REPO_ROOT = current_path

# Configure absolute paths
RAW_DATA_PATH       = REPO_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = REPO_ROOT / "data" / "processed"
CLUSTERING_PATH     = PROCESSED_DATA_PATH / "circuit_clustering"
OUTPUTS_PATH        = REPO_ROOT / "notebooks" / "data_engineering" / "outputs"

# Create outputs directory if it doesn't exist
OUTPUTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"Repository root:    {REPO_ROOT}")
print(f"Raw data:           {RAW_DATA_PATH}")
print(f"Processed data:     {PROCESSED_DATA_PATH}")
print(f"Clustering assets:  {CLUSTERING_PATH}")
print(f"Outputs:            {OUTPUTS_PATH}")

# Sanity check: N03 artifacts exist
assert (CLUSTERING_PATH / "circuit_clusters_k4.parquet").exists(), \
    "ERROR: circuit_clusters_k4.parquet not found. Run N03 first."
assert (CLUSTERING_PATH / "circuit_features_with_clusters_k4.parquet").exists(), \
    "ERROR: circuit_features_with_clusters_k4.parquet not found. Run N03 first."

print("\nN03 artifacts: OK")


Repository root:    c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
Raw data:           c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw
Processed data:     c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed
Clustering assets:  c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\circuit_clustering
Outputs:            c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\notebooks\data_engineering\outputs

N03 artifacts: OK


---

## Step 1: Data Loading & Baseline Filtering

We reload the master dataset using the same `load_all_races()` pattern established in N02 and N03.
Since notebooks cannot import functions from each other, the three loader functions are reproduced here.

### Filters applied to laps
After loading we apply four baseline filters to keep only **clean racing laps**:

| Filter | Reason |
|--------|--------|
| `IsAccurate == True` | Removes laps distorted by Safety Car, red flags, or FastF1 estimation errors |
| `Deleted == False` | Removes laps deleted by stewards (track limits, etc.) |
| `LapTime_s < 180` | Outlier guard — pit-in/out laps and formation laps can exceed 3 min |
| `LapNumber > 1` | Removes formation lap (no racing data) |

We also load the **N03 cluster mapping** here so it is available throughout the notebook.

#### Loader Functions

In [4]:
def load_single_gp(gp_path: Path, year: str, gp_name: str) -> dict:
    try:
        laps_file      = gp_path / "laps.parquet"
        intervals_file = gp_path / "intervals.parquet"
        weather_file   = gp_path / "weather.parquet"
        pitstops_file  = gp_path / "pitstops.parquet"

        if not all([laps_file.exists(), intervals_file.exists(),
                    weather_file.exists(), pitstops_file.exists()]):
            print(f"  WARNING: {gp_name}: Missing files, skipping...")
            return None

        laps_df      = pd.read_parquet(laps_file)
        intervals_df = pd.read_parquet(intervals_file)
        weather_df   = pd.read_parquet(weather_file)
        pitstops_df  = pd.read_parquet(pitstops_file)

        for df in [laps_df, intervals_df, weather_df, pitstops_df]:
            df['GP_Name'] = gp_name
            df['Year']    = int(year)

        return {'laps': laps_df, 'intervals': intervals_df,
                'weather': weather_df, 'pitstops': pitstops_df}

    except Exception as e:
        print(f"  ERROR: {gp_name}: {str(e)}")
        return None


def combine_master_dataframes(all_laps, all_intervals, all_weather, all_pitstops):
    laps_master      = pd.concat(all_laps,      ignore_index=True)
    intervals_master = pd.concat(all_intervals, ignore_index=True)
    weather_master   = pd.concat(all_weather,   ignore_index=True)
    pitstops_master  = pd.concat(all_pitstops,  ignore_index=True)

    laps_master = laps_master.sort_values(
        ['Year', 'GP_Name', 'DriverNumber', 'LapNumber']
    ).reset_index(drop=True)

    intervals_master = intervals_master.sort_values(
        ['Year', 'GP_Name', 'date']
    ).reset_index(drop=True)

    return laps_master, intervals_master, weather_master, pitstops_master


def load_all_races():
    print("Loading all race data...")
    print("=" * 60)

    all_laps, all_intervals, all_weather, all_pitstops = [], [], [], []
    loaded_gps = 0

    for year_path in sorted(RAW_DATA_PATH.glob("*")):
        if not year_path.is_dir():
            continue
        for gp_path in sorted(year_path.glob("*")):
            if not gp_path.is_dir():
                continue
            gp_name  = gp_path.name.replace('_', ' ')
            gp_data  = load_single_gp(gp_path, year_path.name, gp_name)
            if gp_data is None:
                continue
            all_laps.append(gp_data['laps'])
            all_intervals.append(gp_data['intervals'])
            all_weather.append(gp_data['weather'])
            all_pitstops.append(gp_data['pitstops'])
            loaded_gps += 1

    print("=" * 60)
    laps_master, intervals_master, weather_master, pitstops_master = \
        combine_master_dataframes(all_laps, all_intervals, all_weather, all_pitstops)

    print(f"\nLoaded {loaded_gps} GPs")
    print(f"  Laps:      {len(laps_master):,} rows")
    print(f"  Intervals: {len(intervals_master):,} rows")
    print(f"  Weather:   {len(weather_master):,} rows")
    print(f"  Pitstops:  {len(pitstops_master):,} rows")
    print("=" * 60)

    return laps_master, intervals_master, weather_master, pitstops_master


##### Load Data + Filtering 

In [5]:
laps_master, intervals_master, weather_master, pitstops_master = load_all_races()

# Load N03 cluster mapping
circuit_clusters = pd.read_parquet(CLUSTERING_PATH / "circuit_clusters_k4.parquet")
circuit_features = pd.read_parquet(CLUSTERING_PATH / "circuit_features_with_clusters_k4.parquet")

print(f"\nCluster mapping loaded: {len(circuit_clusters)} circuits")
print(circuit_clusters.sort_values('Cluster').to_string(index=False))


Loading all race data...

Loaded 71 GPs
  Laps:      79,032 rows
  Intervals: 1,698,095 rows
  Weather:   11,244 rows
  Pitstops:  2,676 rows

Cluster mapping loaded: 25 circuits
          GP_Name  Cluster
           Suzuka        0
Spa-Francorchamps        0
      Silverstone        0
         Shanghai        0
           Sakhir        0
        Las Vegas        0
           Lusail        0
           Austin        1
            Monza        1
       Yas Island        1
            Miami        1
    Miami Gardens        1
       Marina Bay        1
           Jeddah        1
            Imola        1
             Baku        1
        Melbourne        2
         Montréal        2
        São Paulo        2
        Zandvoort        2
           Monaco        3
         Budapest        3
        Barcelona        3
        Spielberg        3
      Mexico City        3


#### Baseline filtering

In [6]:
# Convert LapTime to seconds for filtering
laps_master['LapTime_s'] = laps_master['LapTime'].dt.total_seconds()

# Apply baseline filters
laps_clean = laps_master[
    (laps_master['IsAccurate'] == True) &
    (laps_master['Deleted']    == False) &
    (laps_master['LapTime_s']  <  180)  &
    (laps_master['LapNumber']  >  1)
].copy().reset_index(drop=True)

# Summary
removed = len(laps_master) - len(laps_clean)
print(f"Original laps:  {len(laps_master):,}")
print(f"Removed:        {removed:,}  ({removed/len(laps_master)*100:.1f}%)")
print(f"Clean laps:     {len(laps_clean):,}")
print(f"\nBy year:")
print(laps_clean.groupby('Year').size().rename('laps').to_string())


Original laps:  79,032
Removed:        10,910  (13.8%)
Clean laps:     68,122

By year:
Year
2023    22106
2024    23256
2025    22760


---

## Step 2: Fuel-Corrected Degradation Features

Raw lap times contain two confounded effects:
- **Tire degradation**: car gets slower as tires wear (+seconds)
- **Fuel burn**: car gets faster as fuel load decreases (-seconds)

To isolate pure tire degradation we apply the fuel correction methodology used in the legacy
tire prediction notebooks (`N01_tire_prediction.ipynb`, `N02_model_tire_predictions.ipynb`):

FuelAdjustedLapTime  = LapTime_s + FuelEffect    # removes the fuel advantage

**Constant:** `0.055 s/lap` — empirical F1 value (midpoint of the 0.05–0.06 s/lap range
documented by Pirelli and independent analysts). Every lap driven, the car burns ~1.5 kg of
fuel, gaining ~0.055 s. Adding this back to the raw lap time gives a fuel-neutral baseline
from which true tire degradation can be measured.

**Baseline:** first valid lap of each stint (TyreLife minimum per driver-stint group).




In [7]:
def add_fuel_corrected_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add fuel-corrected degradation features to lap-level DataFrame.

    Methodology: legacy N01_tire_prediction.ipynb + N02_model_tire_predictions.ipynb
    Constant: 0.055 s/lap (empirical F1 fuel effect, midpoint of 0.05-0.06 s/lap range)

    Features added:
        FuelEffect              - seconds recovered from fuel burn since stint start
        FuelAdjustedLapTime     - LapTime_s with fuel advantage removed
        FuelAdjustedDegAbsolute - seconds slower vs first stint lap (fuel-corrected)
        FuelAdjustedDegPercent  - % slower vs first stint lap (fuel-corrected)
    """
    FUEL_EFFECT_PER_LAP = 0.055  # s/lap

    result = df.copy()
    result['FuelEffect']          = np.nan
    result['FuelAdjustedLapTime'] = np.nan
    result['FuelAdjustedDegAbsolute'] = np.nan
    result['FuelAdjustedDegPercent']  = np.nan

    groups = result.groupby(
        ['Year', 'GP_Name', 'DriverNumber', 'Stint'], sort=False
    )

    for name, group in groups:
        if group['LapTime_s'].isna().all():
            continue

        # Baseline: first lap of the stint
        baseline_tyrelife = group['TyreLife'].min()
        baseline_mask     = group['TyreLife'] == baseline_tyrelife
        baseline_laptime  = group.loc[baseline_mask, 'LapTime_s'].mean()

        if pd.isna(baseline_laptime):
            continue

        # Fuel effect: laps since stint start × 0.055 s/lap
        laps_from_baseline = group['TyreLife'] - baseline_tyrelife
        fuel_effect        = laps_from_baseline * FUEL_EFFECT_PER_LAP
        adjusted           = group['LapTime_s'] + fuel_effect

        result.loc[group.index, 'FuelEffect']              = fuel_effect
        result.loc[group.index, 'FuelAdjustedLapTime']     = adjusted
        result.loc[group.index, 'FuelAdjustedDegAbsolute'] = adjusted - baseline_laptime
        result.loc[group.index, 'FuelAdjustedDegPercent']  = (adjusted / baseline_laptime - 1) * 100

    return result


laps_clean = add_fuel_corrected_features(laps_clean)

# Verification
print("Fuel correction applied.")
print(f"\nNull rates:")
for col in ['FuelEffect', 'FuelAdjustedLapTime', 'FuelAdjustedDegAbsolute', 'FuelAdjustedDegPercent']:
    null_pct = laps_clean[col].isna().mean() * 100
    print(f"  {col:<30s}: {null_pct:.1f}% null")

print(f"\nSanity check — FuelAdjustedLapTime > LapTime_s for TyreLife > 1:")
mask = laps_clean['TyreLife'] > 1
ok   = (laps_clean.loc[mask, 'FuelAdjustedLapTime'] >= laps_clean.loc[mask, 'LapTime_s']).mean()
print(f"  {ok*100:.2f}% of laps pass (expected ~100%)")

print(f"\nSample (one stint):")
sample = laps_clean[
    (laps_clean['GP_Name'] == 'Sakhir') &
    (laps_clean['Year']    == 2023) &
    (laps_clean['DriverNumber'] == 1) &
    (laps_clean['Stint']   == 2)
][['LapNumber','TyreLife','LapTime_s','FuelEffect',
   'FuelAdjustedLapTime','FuelAdjustedDegAbsolute']].head(10)
print(sample.to_string(index=False))


Fuel correction applied.

Null rates:
  FuelEffect                    : 0.7% null
  FuelAdjustedLapTime           : 0.7% null
  FuelAdjustedDegAbsolute       : 0.7% null
  FuelAdjustedDegPercent        : 0.7% null

Sanity check — FuelAdjustedLapTime > LapTime_s for TyreLife > 1:
  100.00% of laps pass (expected ~100%)

Sample (one stint):
Empty DataFrame
Columns: [LapNumber, TyreLife, LapTime_s, FuelEffect, FuelAdjustedLapTime, FuelAdjustedDegAbsolute]
Index: []


In [8]:
# Verify effects in long stint
sample_row = laps_clean[laps_clean['TyreLife'] > 15].iloc[0]
sample = laps_clean[
    (laps_clean['GP_Name']      == sample_row['GP_Name']) &
    (laps_clean['Year']         == sample_row['Year']) &
    (laps_clean['DriverNumber'] == sample_row['DriverNumber']) &
    (laps_clean['Stint']        == sample_row['Stint'])
][['LapNumber','TyreLife','LapTime_s','FuelEffect',
   'FuelAdjustedLapTime','FuelAdjustedDegAbsolute']].head(10)
print(sample.to_string(index=False))


 LapNumber  TyreLife  LapTime_s  FuelEffect  FuelAdjustedLapTime  FuelAdjustedDegAbsolute
    18.000     2.000    100.989       0.000              100.989                    0.000
    19.000     3.000    100.809       0.055              100.864                   -0.125
    20.000     4.000    100.883       0.110              100.993                    0.004
    21.000     5.000    100.980       0.165              101.145                    0.156
    22.000     6.000    101.129       0.220              101.349                    0.360
    23.000     7.000    100.921       0.275              101.196                    0.207
    24.000     8.000    101.203       0.330              101.533                    0.544
    25.000     9.000    101.268       0.385              101.653                    0.664
    26.000    10.000    101.857       0.440              102.297                    1.308
    27.000    11.000    101.795       0.495              102.290                    1.301


---

## Step 3: Sequential Lap Features

Within a stint, consecutive laps are not independent — the previous lap's performance
directly influences the current one. These features capture **momentum and trend**,
which are critical inputs for the XGBoost lap time predictor (v0.4.0).

Ported from `legacy/notebooks/ML_tyre_pred/N00_model_lap_prediction.ipynb`.

| Feature | Description |
|---------|-------------|
| `Prev_LapTime` | Previous lap time (s) |
| `Prev_SpeedI1/I2/FL/ST` | Previous lap speed trap readings (km/h) |
| `Prev_TyreLife` | Previous lap tire age |
| `LapTime_Delta` | Change in lap time vs previous lap (s) |
| `SpeedI1/I2/FL/ST_Delta` | Change in sector speeds vs previous lap (km/h) |
| `LapTime_Trend` | Second derivative of lap time — is pace improving or worsening? |

**NaN policy:** The first lap of each stint has no predecessor → left as `NaN`.
This is semantically correct: NaN signals "no previous lap exists" and XGBoost
handles it natively via its missing value branch. Filling with 0 would imply
the previous lap lasted 0 seconds — a lie the model would learn as signal.
TCN masking and padding are handled in the model-specific notebooks (v0.4.0).

In [9]:
def add_sequential_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add previous-lap and delta features within each driver stint.

    Groups by Year + GP_Name + DriverNumber + Stint, sorted by LapNumber.
    First lap of each stint → NaN (no predecessor exists).
    NaN is left intentionally: XGBoost handles it natively via missing value branch.
    TCN masking handled in model notebooks (v0.4.0).

    Features added: Prev_LapTime, Prev_SpeedI1/I2/FL/ST, Prev_TyreLife,
                    LapTime_Delta, Speed*_Delta, LapTime_Trend
    """
    result = df.copy()
    speed_cols = ['SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST']

    result = result.sort_values(
        ['Year', 'GP_Name', 'DriverNumber', 'Stint', 'LapNumber']
    ).reset_index(drop=True)

    grp = result.groupby(
        ['Year', 'GP_Name', 'DriverNumber', 'Stint'], sort=False
    )

    # Previous values
    result['Prev_LapTime']  = grp['LapTime_s'].shift(1)
    result['Prev_TyreLife'] = grp['TyreLife'].shift(1)
    for col in speed_cols:
        result[f'Prev_{col}'] = grp[col].shift(1)

    # Deltas
    result['LapTime_Delta'] = result['LapTime_s'] - result['Prev_LapTime']
    for col in speed_cols:
        result[f'{col}_Delta'] = result[col] - result[f'Prev_{col}']

    # Trend (second derivative of lap time)
    result['LapTime_Trend'] = grp['LapTime_Delta'].shift(1)
    result['LapTime_Trend'] = result['LapTime_Delta'] - result['LapTime_Trend']

    # No fillna — NaN on first lap of each stint is meaningful signal.
    return result


laps_clean = add_sequential_features(laps_clean)

seq_cols = (
    ['Prev_LapTime', 'Prev_TyreLife', 'LapTime_Delta', 'LapTime_Trend'] +
    [f'Prev_{c}' for c in ['SpeedI1','SpeedI2','SpeedFL','SpeedST']] +
    [f'{c}_Delta' for c in ['SpeedI1','SpeedI2','SpeedFL','SpeedST']]
)
print(f"Sequential features added: {len(seq_cols)}")
print(f"\nNull rates (NaN = first lap of stint or missing speed trap):")
for col in seq_cols:
    pct = laps_clean[col].isna().mean() * 100
    print(f"  {col:<25s}: {pct:.1f}%")

print(f"\nSample:")
sample_row = laps_clean[laps_clean['TyreLife'] > 5].iloc[0]
sample = laps_clean[
    (laps_clean['GP_Name']      == sample_row['GP_Name']) &
    (laps_clean['Year']         == sample_row['Year']) &
    (laps_clean['DriverNumber'] == sample_row['DriverNumber']) &
    (laps_clean['Stint']        == sample_row['Stint'])
][['LapNumber','TyreLife','LapTime_s','Prev_LapTime',
   'LapTime_Delta','LapTime_Trend']].head(6)
print(sample.to_string(index=False))

Sequential features added: 12

Null rates (NaN = first lap of stint or missing speed trap):
  Prev_LapTime             : 5.8%
  Prev_TyreLife            : 5.9%
  LapTime_Delta            : 5.8%
  LapTime_Trend            : 10.9%
  Prev_SpeedI1             : 21.8%
  Prev_SpeedI2             : 6.8%
  Prev_SpeedFL             : 5.8%
  Prev_SpeedST             : 13.8%
  SpeedI1_Delta            : 34.2%
  SpeedI2_Delta            : 6.9%
  SpeedFL_Delta            : 5.8%
  SpeedST_Delta            : 19.9%

Sample:
 LapNumber  TyreLife  LapTime_s  Prev_LapTime  LapTime_Delta  LapTime_Trend
     2.000     2.000    103.009           NaN            NaN            NaN
     3.000     3.000    102.700       103.009         -0.309            NaN
     4.000     4.000    102.623       102.700         -0.077          0.232
     5.000     5.000    102.151       102.623         -0.472         -0.395
     6.000     6.000    102.857       102.151          0.706          1.178
     7.000     7.000    102.81

---

## Step 4: Stint Degradation Rate

With fuel-corrected lap times available, we can now compute the **rate at which tires
degrade** within each stint. This is the key input for the TCN tire degradation model (v0.4.0).

We use a **rolling 3-lap linear regression** (`numpy.polyfit`) to estimate the local slope
of `FuelAdjustedLapTime` vs `TyreLife`. This captures the instantaneous degradation trend
rather than a global stint average, making it robust to early graining and late cliff effects.

| Feature | Description |
|---------|-------------|
| `DegradationRate` | Rolling 3-lap slope of FuelAdjustedLapTime vs TyreLife (s/lap) |
| `CumulativeDeg` | Total seconds lost since stint start (fuel-corrected) |
| `DegAcceleration` | Change in DegradationRate vs previous lap — is deg accelerating? |

**NaN policy:** Lap 1 of each stint (window too small for polyfit) → left as `NaN`.
Same rationale as Step 3: NaN is semantically correct and handled natively by XGBoost.

In [10]:
def add_degradation_rate_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add rolling degradation rate features using 3-lap linear regression per stint.

    Uses FuelAdjustedLapTime (Step 2) to isolate pure tire degradation.
    Requires minimum 2 points per window (min_periods=2).

    Features added: DegradationRate, CumulativeDeg, DegAcceleration

    NaN policy: lap 1 of each stint has no predecessor window → left as NaN.
    XGBoost handles NaN natively via its missing value branch.
    """
    result = df.copy()
    result['DegradationRate'] = np.nan
    result['CumulativeDeg']   = np.nan
    result['DegAcceleration'] = np.nan

    groups = result.groupby(
        ['Year', 'GP_Name', 'DriverNumber', 'Stint'], sort=False
    )

    for name, group in groups:
        idx = group.index
        adj_times  = group['FuelAdjustedLapTime'].values
        tyre_lives = group['TyreLife'].values
        n = len(group)

        if n < 2:
            continue

        # Rolling 3-lap slope (window=3, min_periods=2)
        deg_rates = np.full(n, np.nan)
        for i in range(1, n):          # start at index 1 (need ≥2 points)
            start = max(0, i - 2)      # up to 3 laps back
            x = tyre_lives[start:i+1]
            y = adj_times[start:i+1]
            if len(x) >= 2 and not np.isnan(y).any():
                slope = np.polyfit(x, y, 1)[0]
                deg_rates[i] = slope

        result.loc[idx, 'DegradationRate'] = deg_rates

        # Cumulative degradation since stint start
        base = group['FuelAdjustedDegAbsolute'].iloc[0]
        result.loc[idx, 'CumulativeDeg'] = (
            group['FuelAdjustedDegAbsolute'] - base
        ).values

        # Degradation acceleration (change in rate)
        accel = np.full(n, np.nan)
        for i in range(1, n):
            if not np.isnan(deg_rates[i]) and not np.isnan(deg_rates[i-1]):
                accel[i] = deg_rates[i] - deg_rates[i-1]
        result.loc[idx, 'DegAcceleration'] = accel

    # No fillna — NaN on first lap of each stint is meaningful signal.
    return result


laps_clean = add_degradation_rate_features(laps_clean)

# Summary
print("Degradation rate features added.")
print(f"\nNull rates (NaN = first lap of stint, expected ~5–10%):")
for col in ['DegradationRate', 'CumulativeDeg', 'DegAcceleration']:
    pct = laps_clean[col].isna().mean() * 100
    print(f"  {col:<20s}: {pct:.1f}%")

print(f"\nDegradationRate stats (s/lap):")
print(laps_clean['DegradationRate'].describe().to_string())

print(f"\nSample (same stint as Step 3):")
sample_row = laps_clean[laps_clean['TyreLife'] > 5].iloc[0]
sample = laps_clean[
    (laps_clean['GP_Name']      == sample_row['GP_Name']) &
    (laps_clean['Year']         == sample_row['Year']) &
    (laps_clean['DriverNumber'] == sample_row['DriverNumber']) &
    (laps_clean['Stint']        == sample_row['Stint'])
][['LapNumber','TyreLife','FuelAdjustedLapTime',
   'DegradationRate','CumulativeDeg','DegAcceleration']].head(8)
print(sample.to_string(index=False))


Degradation rate features added.

Null rates (NaN = first lap of stint, expected ~5–10%):
  DegradationRate     : 5.9%
  CumulativeDeg       : 0.8%
  DegAcceleration     : 11.0%

DegradationRate stats (s/lap):
count   64104.000
mean       -0.025
std         1.227
min       -64.635
25%        -0.115
50%         0.041
75%         0.188
max        30.762

Sample (same stint as Step 3):
 LapNumber  TyreLife  FuelAdjustedLapTime  DegradationRate  CumulativeDeg  DegAcceleration
     2.000     2.000              103.009              NaN          0.000              NaN
     3.000     3.000              102.755           -0.254         -0.254              NaN
     4.000     4.000              102.733           -0.138         -0.276            0.116
     5.000     5.000              102.316           -0.219         -0.693           -0.082
     6.000     6.000              103.077            0.172          0.068            0.391
     7.000     7.000              103.092            0.388          

In [11]:
# Inspect how many laps fall outside realistic range
realistic_range = (-2.0, 2.0)  # s/lap — beyond this is noise, not real degradation

outside = (
    (laps_clean['DegradationRate'] < realistic_range[0]) |
    (laps_clean['DegradationRate'] > realistic_range[1])
)
print(f"DegradationRate outside [{realistic_range[0]}, {realistic_range[1]}] s/lap:")
print(f"  {outside.sum()} laps  ({outside.mean()*100:.2f}%)")

# Clip to realistic range
laps_clean['DegradationRate'] = laps_clean['DegradationRate'].clip(*realistic_range)
laps_clean['DegAcceleration'] = laps_clean['DegAcceleration'].clip(*realistic_range)

print(f"\nAfter clipping:")
print(laps_clean['DegradationRate'].describe().to_string())


DegradationRate outside [-2.0, 2.0] s/lap:
  835 laps  (1.23%)

After clipping:
count   64104.000
mean        0.009
std         0.453
min        -2.000
25%        -0.115
50%         0.041
75%         0.188
max         2.000


---

## Step 5: Weather Features

Track and air conditions directly affect tire grip, degradation rate, and lap time.
A SOFT tire at 50°C track temp behaves completely differently than the same tire at 30°C,
yet without weather features the model sees only `TyreLife = 5` in both cases.

| Feature | Description |
|---------|-------------|
| `AirTemp` | Air temperature (°C) — aerodynamic downforce and engine cooling |
| `TrackTemp` | Track surface temperature (°C) — primary driver of tire grip and degradation |
| `Humidity` | Relative humidity (%) — influences tire rubber chemistry and brake cooling |
| `Rainfall` | Binary flag (0=dry, 1=wet) — rain lap, major performance discontinuity |

**Merge strategy:** FastF1 weather data is sampled every ~30–60 s (session time).
We align each lap to its closest weather reading using `pd.merge_asof(direction='nearest')`
per race, joining on session `Time` (timedelta from session start).

> **Note on Rainfall:** Our baseline filter (`IsAccurate == True`) already excludes most
> rain and safety car laps — so virtually all surviving laps have `Rainfall == 0`.
> The feature is kept for robustness in future inference scenarios (e.g., light drizzle
> that didn't trigger a full SC) and because the legacy notebooks confirmed it as a
> significant predictor when present.

In [12]:
def add_weather_features(laps_df: pd.DataFrame, weather_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge weather conditions to lap-level data using nearest-time join.

    For each race (Year + GP_Name), matches each lap to the closest weather
    sample using pd.merge_asof(direction='nearest') on session Time.

    Features added: AirTemp, TrackTemp, Humidity, Rainfall

    NaN policy: Rainfall filled with 0 (no rain); other weather cols left NaN
    if no weather data available for a race (rare edge case).
    """
    WEATHER_COLS = ['AirTemp', 'TrackTemp', 'Humidity', 'Rainfall']
    result = laps_df.copy()

    for col in WEATHER_COLS:
        result[col] = np.nan

    for (year, gp), lap_group in result.groupby(['Year', 'GP_Name']):
        wth = (
            weather_df[
                (weather_df['Year'] == year) & (weather_df['GP_Name'] == gp)
            ][['Time'] + WEATHER_COLS]
            .dropna(subset=['Time'])
            .sort_values('Time')
        )
        if wth.empty:
            continue

        # Sort laps by session time for merge_asof
        laps_sorted = lap_group[['Time']].sort_values('Time')

        # Nearest-time join: each lap gets the closest weather sample
        merged = pd.merge_asof(
            laps_sorted, wth, on='Time', direction='nearest'
        )
        merged.index = laps_sorted.index

        for col in WEATHER_COLS:
            result.loc[merged.index, col] = merged[col].values

    # Rainfall: fill NaN as 0 (no rain) and cast to int flag
    result['Rainfall'] = result['Rainfall'].fillna(0).astype(int)

    return result


laps_clean = add_weather_features(laps_clean, weather_master)

# Summary
print("Weather features added.")
print(f"\nNull rates:")
for col in ['AirTemp', 'TrackTemp', 'Humidity', 'Rainfall']:
    pct = laps_clean[col].isna().mean() * 100
    print(f"  {col:<12s}: {pct:.1f}%")

print(f"\nWeather stats (°C / %):")
print(laps_clean[['AirTemp', 'TrackTemp', 'Humidity']].describe().to_string())

print(f"\nRainfall distribution (0=dry, 1=wet):")
print(laps_clean['Rainfall'].value_counts().sort_index().to_string())


Weather features added.

Null rates:
  AirTemp     : 0.0%
  TrackTemp   : 0.0%
  Humidity    : 0.0%
  Rainfall    : 0.0%

Weather stats (°C / %):
        AirTemp  TrackTemp  Humidity
count 68122.000  68122.000 68122.000
mean     23.911     35.222    52.926
std       4.603      8.736    16.024
min      14.200     16.700    17.000
25%      20.200     29.600    42.000
50%      24.200     34.200    55.000
75%      27.300     42.700    64.000
max      33.800     52.700    92.000

Rainfall distribution (0=dry, 1=wet):
Rainfall
0    66136
1     1986


---

## Step 6: Race Context Features

All features in this step derive directly from existing columns — no joins needed.

| Feature | Source | Description |
|---------|--------|-------------|
| `Sector1_s` / `Sector2_s` / `Sector3_s` | `Sector1Time` etc. | Sector times as float (seconds) |
| `race_phase` | `LapNumber` / `MaxLap` | early / mid / late (terciles) |
| `laps_remaining` | `LapNumber` / `MaxLap` | Laps left in the race |
| `track_status_clean` | `TrackStatus` | 0=clear, 1=yellow/VSC, 2=SC/red flag |


In [13]:
def add_race_context_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add race context features derivable directly from existing columns.
    No joins or external data required.
    """
    result = df.copy()

    # --- Sector times as float (seconds) ---
    for col, new_col in [('Sector1Time', 'Sector1_s'),
                         ('Sector2Time', 'Sector2_s'),
                         ('Sector3Time', 'Sector3_s')]:
        if col in result.columns:
            result[new_col] = result[col].dt.total_seconds()

    # --- Max laps per race ---
    max_laps = (
        result.groupby(['Year', 'GP_Name'])['LapNumber']
        .transform('max')
    )

    # --- Race phase (early / mid / late) ---
    lap_fraction = result['LapNumber'] / max_laps
    result['race_phase'] = pd.cut(
        lap_fraction,
        bins=[0, 0.33, 0.67, 1.0],
        labels=['early', 'mid', 'late'],
        include_lowest=True
    )

    # --- Laps remaining ---
    result['laps_remaining'] = (max_laps - result['LapNumber']).astype(int)

    # --- Track status simplified ---
    # FastF1 codes: 1=clear, 2=yellow, 3=SC deployed, 4=SC ending,
    #               5=red flag, 6=VSC deployed, 7=VSC ending
    status_map = {1: 0, 2: 1, 3: 2, 4: 2, 5: 2, 6: 1, 7: 1}
    result['track_status_clean'] = (
        result['TrackStatus'].map(status_map).fillna(0).astype(int)
    )

    return result


laps_clean = add_race_context_features(laps_clean)

# Summary
print("Race context features added.")
print(f"\nrace_phase distribution:")
print(laps_clean['race_phase'].value_counts().sort_index().to_string())

print(f"\ntrack_status_clean distribution:")
status_labels = {0: 'clear', 1: 'yellow/VSC', 2: 'SC/red'}
counts = laps_clean['track_status_clean'].value_counts().sort_index()
for k, v in counts.items():
    print(f"  {k} ({status_labels[k]}): {v:,} laps ({v/len(laps_clean)*100:.1f}%)")

print(f"\nlaps_remaining range: {laps_clean['laps_remaining'].min()} – {laps_clean['laps_remaining'].max()}")
print(f"Null rates:")
for col in ['Sector1_s', 'Sector2_s', 'Sector3_s', 'race_phase', 'laps_remaining', 'track_status_clean']:
    pct = laps_clean[col].isna().mean() * 100
    print(f"  {col:<22s}: {pct:.1f}%")


Race context features added.

race_phase distribution:
race_phase
early    21091
mid      23613
late     23418

track_status_clean distribution:
  0 (clear): 68,122 laps (100.0%)

laps_remaining range: 0 – 76
Null rates:
  Sector1_s             : 0.0%
  Sector2_s             : 0.0%
  Sector3_s             : 0.0%
  race_phase            : 0.0%
  laps_remaining        : 0.0%
  track_status_clean    : 0.0%


---

## Step 7: Circuit-Cluster Features

Merges the N03 K-Means artifacts into the lap-level dataset.
`Cluster` encodes circuit archetype; the other two features normalize
lap time and speed to the circuit's characteristic profile.


In [14]:
def add_cluster_features(df: pd.DataFrame,
                         clusters: pd.DataFrame,
                         circuit_features: pd.DataFrame) -> pd.DataFrame:
    """
    Merge N03 circuit clustering artifacts into lap-level data.

    Features added:
        Cluster                  - Circuit archetype (0-3) from K-Means
        lap_time_vs_cluster_mean - Delta from cluster mean lap time (s)
        mean_sector_speed        - Circuit-level mean sector speed (km/h)
    """
    result = df.copy()

    # Cluster mean lap time (from circuit_features)
    cluster_mean = (
        circuit_features.groupby('Cluster')['mean_laptime']
        .mean()
        .reset_index()
        .rename(columns={'mean_laptime': 'cluster_mean_laptime'})
    )

    # Merge Cluster + mean_sector_speed on GP_Name
    circuit_lookup = clusters.merge(
        circuit_features[['GP_Name', 'mean_sector_speed']],
        on='GP_Name', how='left'
    )
    result = result.merge(circuit_lookup, on='GP_Name', how='left')

    # Merge cluster mean lap time on Cluster
    result = result.merge(cluster_mean, on='Cluster', how='left')

    # Delta from cluster mean
    result['lap_time_vs_cluster_mean'] = result['LapTime_s'] - result['cluster_mean_laptime']

    # Drop helper column
    result = result.drop(columns=['cluster_mean_laptime'])

    return result


laps_clean = add_cluster_features(laps_clean, circuit_clusters, circuit_features)

# Summary
print("Cluster features added.")
print(f"\nCluster distribution:")
print(laps_clean['Cluster'].value_counts().sort_index().rename({
    0: '0 - Strategic Chaos',
    1: '1 - Long High-Speed',
    2: '2 - Warm Conservative',
    3: '3 - High-Stress Outliers'
}).to_string())

print(f"\nNull rates:")
for col in ['Cluster', 'lap_time_vs_cluster_mean', 'mean_sector_speed']:
    pct = laps_clean[col].isna().mean() * 100
    print(f"  {col:<28s}: {pct:.1f}%")

print(f"\nlap_time_vs_cluster_mean stats (s):")
print(laps_clean['lap_time_vs_cluster_mean'].describe().to_string())


Cluster features added.

Cluster distribution:
Cluster
0 - Strategic Chaos         16029
1 - Long High-Speed         21494
2 - Warm Conservative       11844
3 - High-Stress Outliers    17557

Null rates:
  Cluster                     : 1.8%
  lap_time_vs_cluster_mean    : 1.8%
  mean_sector_speed           : 1.8%

lap_time_vs_cluster_mean stats (s):
count   66924.000
mean       -2.462
std         6.943
min       -18.541
25%        -7.833
50%        -2.524
75%         1.766
max        67.125


In [15]:
# Fix: map 'Spain' → 'Barcelona' cluster (duplicate artefact from N01)
spain_mask = laps_clean['GP_Name'] == 'Spain'
print(f"Spain laps sin cluster: {spain_mask.sum()}")

if spain_mask.sum() > 0:
    barcelona_cluster = circuit_clusters.loc[
        circuit_clusters['GP_Name'] == 'Barcelona', 'Cluster'
    ].iloc[0]
    barcelona_speed = circuit_features.loc[
        circuit_features['GP_Name'] == 'Barcelona', 'mean_sector_speed'
    ].iloc[0]
    cluster_mean_barcelona = (
        circuit_features.groupby('Cluster')['mean_laptime']
        .mean()[barcelona_cluster]
    )

    laps_clean.loc[spain_mask, 'Cluster']           = barcelona_cluster
    laps_clean.loc[spain_mask, 'mean_sector_speed'] = barcelona_speed
    laps_clean.loc[spain_mask, 'lap_time_vs_cluster_mean'] = (
        laps_clean.loc[spain_mask, 'LapTime_s'] - cluster_mean_barcelona
    )

# Verify
print(f"Nulls remaining in Cluster: {laps_clean['Cluster'].isna().sum()}")
print(f"Total laps with cluster: {laps_clean['Cluster'].notna().sum():,}")


Spain laps sin cluster: 1198
Nulls remaining in Cluster: 0
Total laps with cluster: 68,122


---

## Step 7.5: Temporal Normalization — `lap_time_pct_of_race_fastest`

### Por qué la añadimos

F1 mejora ~0.5–1.5 s/vuelta en el mismo circuito de un año a otro por evolución del reglamento,
desarrollo aerodinámico y mejoras en la unidad de potencia. Sin corrección, una vuelta de 92.0 s
en Sakhir 2023 y una de 91.0 s en Sakhir 2024 parecen **eventos distintos** para el modelo,
cuando en realidad representan un rendimiento equivalente —la diferencia de 1 s es drift inter-anual,
no señal de degradación ni estrategia.

`lap_time_pct_of_race_fastest` normaliza cada vuelta como **ratio respecto a la vuelta más rápida
limpia de esa carrera**:

```
lap_time_pct_of_race_fastest = LapTime_s / min(LapTime_s en ese GP y año)
```

- Valor de **1.000** = la vuelta más rápida de la carrera
- Valor de **1.050** = 5 % más lento que la mejor vuelta vista
- El ratio es **agnóstico al año**: un 5 % más lento en 2023 y un 5 % más lento en 2025
  son genuinamente comparables porque ambos se miden contra su propio año

El modelo continúa prediciendo **segundos absolutos** (`LapTime_s` sigue siendo el target).
Esta feature es solo una entrada adicional que ayuda al modelo a comprender la posición
relativa de la vuelta dentro de la carrera.

### Fuentes que respaldan esta feature

| Fuente | Qué valida |
|--------|-----------|
| **Liu et al. (2023)** — arXiv:2304.01512, §4.1 | Demuestra que la normalización relativa intra-serie (ratio vs. referencia de la misma sesión) reduce el error de GFMs bajo drift gradual hasta un 8 %. Nuestra feature aplica la misma lógica al nivel de vuelta individual. |
| **Tapan Babbar (Medium) · Jared-Chan/f1ml (GitHub)** | Patrón documentado en la comunidad F1-ML: normalizar `LapTime_s` respecto al mejor tiempo de la sesión es la forma más robusta de comparar vueltas entre temporadas y circuitos. |
| **concept_drift_strategy.md — Acción 1** | Esta feature es la implementación directa de la Acción 1 de la estrategia anti-drift del proyecto. |

In [16]:
def add_temporal_normalization_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add lap_time_pct_of_race_fastest: ratio of each lap time to the
    fastest clean lap in the same race (Year + GP_Name).

    Formula: LapTime_s / min(LapTime_s per race)

    Purpose: normalizes inter-year pace drift (~0.5–1.5 s/lap/year on same circuit)
    so the model can compare equivalent race pace across seasons without
    conflating year-over-year car development with tire degradation signal.

    References:
    - Liu et al. (2023) arXiv:2304.01512 §4.1 — relative intra-series normalization
      reduces GFM error under gradual concept drift up to 8%.
    - concept_drift_strategy.md — Acción 1
    """
    result = df.copy()

    race_fastest = (
        result.groupby(['Year', 'GP_Name'])['LapTime_s']
        .transform('min')
    )

    result['lap_time_pct_of_race_fastest'] = result['LapTime_s'] / race_fastest

    return result


laps_clean = add_temporal_normalization_features(laps_clean)

# Verification
print("Temporal normalization feature added.")
print(f"\nlap_time_pct_of_race_fastest stats:")
print(laps_clean['lap_time_pct_of_race_fastest'].describe().to_string())
print(f"\nNull rate: {laps_clean['lap_time_pct_of_race_fastest'].isna().mean()*100:.1f}%")

# Sanity check: all values >= 1.0 (no lap can be faster than the race minimum)
under_one = (laps_clean['lap_time_pct_of_race_fastest'] < 1.0).sum()
print(f"Values < 1.0 (should be 0): {under_one}")

# Cross-year comparison: same circuit, same % value = same relative pace
print(f"\nCross-year comparison — Sakhir (fastest 5 laps per year):")
sakhir = laps_clean[laps_clean['GP_Name'] == 'Sakhir'].copy()
for year in sorted(sakhir['Year'].unique()):
    top = sakhir[sakhir['Year'] == year].nsmallest(5, 'LapTime_s')[
        ['Driver', 'LapTime_s', 'lap_time_pct_of_race_fastest']
    ]
    print(f"\n  {year}:")
    print(top.to_string(index=False))

Temporal normalization feature added.

lap_time_pct_of_race_fastest stats:
count   68122.000
mean        1.047
std         0.038
min         1.000
25%         1.028
50%         1.040
75%         1.054
max         1.939

Null rate: 0.0%
Values < 1.0 (should be 0): 0

Cross-year comparison — Sakhir (fastest 5 laps per year):

  2023:
Driver  LapTime_s  lap_time_pct_of_race_fastest
   ZHO     93.996                         1.000
   GAS     95.068                         1.011
   NOR     95.822                         1.019
   SAR     96.037                         1.022
   GAS     96.041                         1.022

  2024:
Driver  LapTime_s  lap_time_pct_of_race_fastest
   VER     92.608                         1.000
   LEC     94.090                         1.016
   VER     94.151                         1.017
   ALO     94.199                         1.017
   VER     94.239                         1.018

  2025:
Driver  LapTime_s  lap_time_pct_of_race_fastest
   PIA     95.140       

---

## Step 8: Feature Audit & Dataset Assembly

All feature engineering steps are complete. This final step audits the full
feature set for null rates and saves the enriched dataset to disk.

The resulting `laps_featured.parquet` is the single input for all ML model
notebooks in v0.4.0. Train/val splits and correlation analysis will be handled
in each model's dedicated EDA notebook, where they can be tailored to the
specific target variable.

**Columns dropped here:** raw FastF1 timedelta columns (`Sector1Time`,
`LapStartTime`, `PitInTime`, etc.) that were either already converted to
seconds during this notebook or are telemetry timestamps not useful as ML features.


In [17]:
# --- Drop columns not needed for ML ---
# 1. Raw FastF1 timedeltas already converted to float seconds
# 2. Redundant columns: LapTime (have LapTime_s), TrackStatus (have track_status_clean)
# 3. No-variance post-filter: Deleted, IsAccurate (always False/True after filter)
# 4. Pure metadata not used as features: FastF1Generated, IsPersonalBest, DeletedReason
cols_to_drop = [c for c in [
    # Raw timedelta columns (converted to *_s during this notebook)
    'Time', 'PitOutTime', 'PitInTime',
    'Sector1Time', 'Sector2Time', 'Sector3Time',
    'LapStartTime', 'LapStartDate',
    'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
    'Time_s',
    # Superseded by derived columns
    'LapTime',          # -> LapTime_s
    'TrackStatus',      # -> track_status_clean
    # Zero variance after baseline filter (all True / all False)
    'Deleted', 'DeletedReason', 'IsAccurate',
    # Metadata not useful as ML features
    'FastF1Generated', 'IsPersonalBest',
] if c in laps_clean.columns]

laps_featured = laps_clean.drop(columns=cols_to_drop)

# --- Fix dtypes ---
laps_featured['Cluster']      = laps_featured['Cluster'].astype(int)
laps_featured['DriverNumber'] = laps_featured['DriverNumber'].astype(int)

# --- Feature audit ---
print(f"{'='*60}")
print(f"FEATURE AUDIT - {laps_featured.shape[1]} columns, {laps_featured.shape[0]:,} rows")
print(f"{'='*60}")
print(f"{'Column':<35} {'Dtype':<15} {'Null %':>7}")
print(f"{'-'*60}")
for col in laps_featured.columns:
    null_pct = laps_featured[col].isna().mean() * 100
    flag = " <- WARNING" if null_pct > 15 else ""
    print(f"{col:<35} {str(laps_featured[col].dtype):<15} {null_pct:>6.1f}%{flag}")

print(f"\nBy year:")
print(laps_featured.groupby('Year').size().rename('laps').to_string())

# --- Save: combined + per-year ---
# Combined (all seasons)
out_combined = PROCESSED_DATA_PATH / "laps_featured.parquet"
laps_featured.to_parquet(out_combined, index=False)
print(f"\nSaved -> {out_combined.relative_to(REPO_ROOT)}  ({laps_featured.shape[0]:,} rows)")

# Per-year splits — useful for model notebooks that train on one season at a time
for year, group in laps_featured.groupby('Year'):
    out_year = PROCESSED_DATA_PATH / f"laps_featured_{year}.parquet"
    group.reset_index(drop=True).to_parquet(out_year, index=False)
    print(f"Saved -> {out_year.relative_to(REPO_ROOT)}  ({len(group):,} rows)")

print(f"\nAll files saved. Final shape: {laps_featured.shape}")


FEATURE AUDIT - 53 columns, 68,122 rows
Column                              Dtype            Null %
------------------------------------------------------------
Driver                              object             0.0%
DriverNumber                        int32              0.0%
LapNumber                           float64            0.0%
Stint                               float64            0.6%
SpeedI1                             float64           17.2% <- WARNING
SpeedI2                             float64            1.1%
SpeedFL                             float64            0.0%
SpeedST                             float64            8.6%
Compound                            object             0.0%
TyreLife                            float64            0.7%
FreshTyre                           bool               0.0%
Team                                object             0.0%
Position                            float64            0.0%
CompoundID                          float64     

---

## Step 9: 2025 Feature Engineering (Inference Mode)

This section runs the exact same feature engineering pipeline on the 2025 season,
producing a held-out test dataset that the models will never see during training.

**Key differences vs Steps 1–8:**
- Data loaded from `data/raw/2025/` only
- Cluster assignment from saved model (`circuit_clusters_k4_2025.parquet`) — never re-fitted
- All feature engineering functions are identical (same constants, same logic)
- Output saved separately: `laps_featured_2025.parquet`

**Prerequisite:** Run N01 Step 7 and N03 Step 7 first.

In [18]:
raw_2025 = RAW_DATA_PATH / "2025"
# Use the stable 2023-2024 cluster mapping — NOT the 2025-specific one.
# K-Means was fitted on 2023-2024 circuit features. Applying it to 2025 features
# via kmeans.predict() causes 7 circuits to shift clusters (e.g. Shanghai C0→C1,
# Barcelona C3→C1) because race-performance features (mean_laptime, etc.) drift
# year-to-year. Using the same lookup ensures the Cluster feature has a consistent
# meaning across 2023, 2024, and 2025 — which is critical for the XGBoost model.
clusters_2025_path = CLUSTERING_PATH / "circuit_clusters_k4.parquet"
features_2025_path = CLUSTERING_PATH / "circuit_features_with_clusters_k4.parquet"

# Canonical name mapping — keep in sync with N01 CIRCUIT_NAME_ALIASES
GP_NAME_ALIASES_2025 = {
    'Miami Gardens': 'Miami',
    'Miami_Gardens': 'Miami',
}

if not raw_2025.exists() or not any(raw_2025.iterdir()):
    print("WARNING: No 2025 data found. Run N01 Step 7 first.")
elif not clusters_2025_path.exists():
    print("WARNING: circuit_clusters_k4.parquet not found. Run N03 first.")
else:
    # ── 1. Load raw 2025 data ─────────────────────────────────────────────────
    all_laps_25, all_int_25, all_wth_25, all_pit_25 = [], [], [], []
    for gp_path in sorted(raw_2025.glob("*")):
        if not gp_path.is_dir():
            continue
        gp_data = load_single_gp(gp_path, "2025", gp_path.name.replace("_", " "))
        if gp_data is None:
            continue
        all_laps_25.append(gp_data['laps'])
        all_int_25.append(gp_data['intervals'])
        all_wth_25.append(gp_data['weather'])
        all_pit_25.append(gp_data['pitstops'])

    laps_25, _, wth_25_master, _ = combine_master_dataframes(
        all_laps_25, all_int_25, all_wth_25, all_pit_25
    )

    # Normalize GP names to match training set canonical names
    laps_25['GP_Name'] = laps_25['GP_Name'].replace(GP_NAME_ALIASES_2025)

    print(f"2025 raw laps loaded: {len(laps_25):,} across {laps_25['GP_Name'].nunique()} GPs")

    # ── 2. Baseline filtering (same as Step 1) ────────────────────────────────
    laps_25['LapTime_s'] = laps_25['LapTime'].dt.total_seconds()
    laps_25_clean = laps_25[
        (laps_25['IsAccurate'] == True) &
        (laps_25['Deleted']    == False) &
        (laps_25['LapTime_s']  <  180)  &
        (laps_25['LapNumber']  >  1)
    ].copy().reset_index(drop=True)

    removed = len(laps_25) - len(laps_25_clean)
    print(f"After filter: {len(laps_25_clean):,} laps  "
          f"(removed {removed:,} = {removed/len(laps_25)*100:.1f}%)")

    # ── 3–7.5. Feature engineering (reuse functions defined above) ────────────
    laps_25_clean = add_fuel_corrected_features(laps_25_clean)
    laps_25_clean = add_sequential_features(laps_25_clean)
    laps_25_clean = add_degradation_rate_features(laps_25_clean)

    # Clip DegradationRate outliers (same range as training)
    laps_25_clean['DegradationRate'] = laps_25_clean['DegradationRate'].clip(-2.0, 2.0)
    laps_25_clean['DegAcceleration'] = laps_25_clean['DegAcceleration'].clip(-2.0, 2.0)

    # ── 5. Weather features ───────────────────────────────────────────────────
    laps_25_clean = add_weather_features(laps_25_clean, wth_25_master)

    laps_25_clean = add_race_context_features(laps_25_clean)

    # ── 7. Circuit clusters — stable 2023-2024 mapping (no refit, no predict) ─
    circuit_clusters_25  = pd.read_parquet(clusters_2025_path)
    circuit_features_25  = pd.read_parquet(features_2025_path)
    laps_25_clean = add_cluster_features(laps_25_clean, circuit_clusters_25, circuit_features_25)

    # Verify cluster consistency: all 2025 circuits should have same Cluster as 2024
    print(f"\nCluster assignments for 2025 circuits (using stable 2023-2024 mapping):")
    cluster_check = (
        laps_25_clean.groupby('GP_Name')['Cluster']
        .first()
        .sort_values()
        .reset_index()
    )
    print(cluster_check.to_string(index=False))

    # ── 7.5. Temporal normalization (same function, ratio within each 2025 race) ─
    laps_25_clean = add_temporal_normalization_features(laps_25_clean)

    # ── 8. Assembly — same drops + dtype fixes as Step 8 ─────────────────────
    cols_to_drop_25 = [c for c in [
        'Time', 'PitOutTime', 'PitInTime',
        'Sector1Time', 'Sector2Time', 'Sector3Time',
        'LapStartTime', 'LapStartDate',
        'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
        'Time_s', 'LapTime', 'TrackStatus',
        'Deleted', 'DeletedReason', 'IsAccurate',
        'FastF1Generated', 'IsPersonalBest',
    ] if c in laps_25_clean.columns]

    laps_25_featured = laps_25_clean.drop(columns=cols_to_drop_25)
    laps_25_featured['Cluster']      = laps_25_featured['Cluster'].astype(int)
    laps_25_featured['DriverNumber'] = laps_25_featured['DriverNumber'].astype(int)

    # ── 9. Audit + save ───────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"2025 FEATURE AUDIT — {laps_25_featured.shape[1]} columns, "
          f"{laps_25_featured.shape[0]:,} rows")
    print(f"{'='*60}")
    print(f"{'Column':<35} {'Dtype':<15} {'Null %':>7}")
    print(f"{'-'*60}")
    for col in laps_25_featured.columns:
        null_pct = laps_25_featured[col].isna().mean() * 100
        flag = " <- WARNING" if null_pct > 15 else ""
        print(f"{col:<35} {str(laps_25_featured[col].dtype):<15} {null_pct:>6.1f}%{flag}")

    out_25 = PROCESSED_DATA_PATH / "laps_featured_2025.parquet"
    laps_25_featured.to_parquet(out_25, index=False)
    print(f"\nSaved -> {out_25.relative_to(REPO_ROOT)}  "
          f"({laps_25_featured.shape[0]:,} rows, {laps_25_featured.shape[1]} cols)")


2025 raw laps loaded: 26,692 across 24 GPs
After filter: 22,760 laps  (removed 3,932 = 14.7%)

Cluster assignments for 2025 circuits (using stable 2023-2024 mapping):
          GP_Name  Cluster
           Suzuka        0
Spa-Francorchamps        0
      Silverstone        0
         Shanghai        0
           Sakhir        0
        Las Vegas        0
           Lusail        0
           Austin        1
            Monza        1
       Yas Island        1
            Miami        1
       Marina Bay        1
           Jeddah        1
            Imola        1
             Baku        1
        Melbourne        2
         Montréal        2
        São Paulo        2
        Zandvoort        2
           Monaco        3
         Budapest        3
        Barcelona        3
        Spielberg        3
      Mexico City        3

2025 FEATURE AUDIT — 53 columns, 22,760 rows
Column                              Dtype            Null %
----------------------------------------------------